# CNN-EXP-042 — WIDTH256, продолжение 10k → 25k

## Главная гипотеза

EXP040 WIDTH256 мог быть недообучен на 10k steps так же, как WIDTH128.

Поэтому архитектуру не меняем:

```text
channels = 256
RF = 1029
8 residual blocks
batch = 4
lr = 1e-3
presence + intensity objectives
тот же sampler / split
```

Берём **точный checkpoint EXP040 @10k** и продолжаем обучение до:

```text
15k
20k
25k
```

Сохраняем checkpoints на всех трёх точках.

Главное сравнение:

```text
WIDTH256 @10k  (EXP040)
vs
WIDTH256 @25k  (EXP042)
```

А затем:

```text
WIDTH128 @25k  (EXP041)
vs
WIDTH256 @25k  (EXP042)
```

**LR намеренно оставляем 1e-3.**
Сейчас тестируем только training budget.


In [1]:
from pathlib import Path
import copy
import gc
import json
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch_directml

from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA — frozen relative to reference
# ============================================================

split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(
    split.intervals("train")["contig"]
)

validation_contigs = set(
    split.intervals("validation")["contig"]
)

assert train_contigs.isdisjoint(
    validation_contigs
)

display(split.summary)

print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(
    batch_size=4
)

print()
print("Input:", train_batch.x.shape)
print("Both orientations:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Positive position-strand:", int(train_batch.target.sum()))


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3

Input: torch.Size([4, 5, 1284])
Both orientations: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Positive position-strand: 33


## Проверка архитектуры

| Stage | Channels | RF |
|---|---:|---:|
| Stem | 5 → 256 | 1 |
| Block 1, k7 d1 | 256 → 256 | 13 |
| Block 2, k3 d2 | 256 → 256 | 21 |
| Block 3, k3 d4 | 256 → 256 | 37 |
| Block 4, k3 d8 | 256 → 256 | 69 |
| Block 5, k3 d16 | 256 → 256 | 133 |
| Block 6, k3 d32 | 256 → 256 | 261 |
| Block 7, k3 d64 | 256 → 256 | 517 |
| Block 8, k3 d128 | 256 → 256 | **1029** |
| Profile head | 256 → 2 | 1029 |

Ожидаемое число параметров: **3,684,354**.

Меняем только ширину representation.


In [4]:
# ============================================================
# ARCHITECTURE ASSERT
# ============================================================

EXPECTED_CHANNELS = 256
EXPECTED_RF = 1029
EXPECTED_PARAMETER_COUNT = 3_684_354

architecture_model = create_cnn_presence_intensity(channels=256)

stem = architecture_model.backbone.stem

actual_channels = int(
    stem.out_channels
)

parameter_count = sum(
    p.numel()
    for p in architecture_model.parameters()
    if p.requires_grad
)

# Compute theoretical RF from actual residual blocks.
rf = 1

for block in architecture_model.backbone.blocks:
    for conv in (block.conv1, block.conv2):
        k = int(conv.kernel_size[0])
        d = int(conv.dilation[0])

        rf += (
            k - 1
        ) * d

print("Stem:", stem.in_channels, "→", stem.out_channels)
print("Residual blocks:", len(architecture_model.backbone.blocks))
print("Theoretical RF:", rf)
print("Profile head:", architecture_model.profile_head.in_channels, "→", architecture_model.profile_head.out_channels)
print("Trainable parameters:", f"{parameter_count:,}")

assert actual_channels == EXPECTED_CHANNELS, (
    f"Сейчас model.py всё ещё создаёт {actual_channels} channels, "
    f"а нужен {EXPECTED_CHANNELS}. "
    "Проверь default channels в create_cnn_baseline() "
    "и create_cnn_presence_intensity(channels=256)."
)

assert rf == EXPECTED_RF
assert architecture_model.profile_head.in_channels == EXPECTED_CHANNELS
assert architecture_model.profile_head.out_channels == 2
assert parameter_count == EXPECTED_PARAMETER_COUNT

del architecture_model
gc.collect()

print()
print("ARCHITECTURE CHECK: OK")


Stem: 5 → 256
Residual blocks: 8
Theoretical RF: 1029
Profile head: 256 → 2
Trainable parameters: 3,684,354

ARCHITECTURE CHECK: OK


In [5]:
# ============================================================
# ONE-STEP DIRECTML / GRADIENT SMOKE
# ============================================================

smoke_model = (
    create_cnn_presence_intensity(channels=256)
    .to(device)
)

smoke_model.train()

smoke_optimizer = DirectMLAdam(
    smoke_model.parameters(),
    lr=1e-3,
)

smoke_batch = train_generator.sample_batch(
    batch_size=4
)

x_gpu = smoke_batch.x_both_strands.to(
    device
)

plus_target_gpu = (
    smoke_batch.plus_target
    .to(device)
)

minus_target_reverse_gpu = (
    smoke_batch.minus_target_reverse
    .to(device)
)

plus_presence_weight_gpu = (
    smoke_batch.plus_loss_weight
    .to(device)
)

minus_presence_weight_reverse_gpu = (
    smoke_batch.minus_loss_weight_reverse
    .to(device)
)

plus_count = (
    smoke_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    smoke_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target_gpu = (
    torch.log1p(plus_count)
    .to(device)
)

minus_intensity_target_reverse_gpu = (
    torch.log1p(minus_count_reverse)
    .to(device)
)

plus_intensity_weight_gpu = (
    2.0
    * smoke_batch.plus_loss_weight
    * (plus_count > 0).float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * smoke_batch.minus_loss_weight_reverse
    * (minus_count_reverse > 0).float()
).to(device)

smoke_optimizer.zero_grad(
    set_to_none=True
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        smoke_model,
        x_gpu,
        batch_size=4,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    smoke_loss = (
        presence_loss
        + 0.1 * intensity_loss
    )

    smoke_loss.backward()
    smoke_optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = float(
    smoke_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .detach()
    .cpu()
    .item()
)

intensity_gradient = float(
    smoke_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .detach()
    .cpu()
    .item()
)

print("Presence loss:", float(presence_loss.detach().cpu().item()))
print("Intensity loss:", float(intensity_loss.detach().cpu().item()))
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

del (
    smoke_model,
    smoke_optimizer,
    smoke_batch,
    x_gpu,
    plus_target_gpu,
    minus_target_reverse_gpu,
    plus_presence_weight_gpu,
    minus_presence_weight_reverse_gpu,
    plus_intensity_target_gpu,
    minus_intensity_target_reverse_gpu,
    plus_intensity_weight_gpu,
    minus_intensity_weight_reverse_gpu,
    plus_presence,
    minus_presence_reverse,
    plus_intensity,
    minus_intensity_reverse,
    presence_loss,
    intensity_loss,
    smoke_loss,
)

gc.collect()

print("DIRECTML SMOKE: OK")


Presence loss: 0.7473300695419312
Intensity loss: 0.5023272037506104
Presence gradient: 13.631284713745117
Intensity gradient: 8.855330467224121
CPU fallback: []
DIRECTML SMOKE: OK


## Продолжение обучения

Это exact resume, а не новый запуск с нуля.

Из EXP040 @10k восстанавливаем:

```text
model_state_dict
optimizer_state_dict
generator RNG state
torch RNG state
training history
EMA losses
seen_position_strand
```

И продолжаем:

```text
10,001 → 25,000
```

Сохраняем:

```text
15k
20k
25k
```

Не меняем:
- LR = 1e-3;
- batch = 4;
- loss;
- sampling;
- seed;
- WIDTH256;
- RF1029.


In [6]:
# ============================================================
# EXPERIMENT CONFIG
# ============================================================

EXPERIMENT_ID = "CNN-EXP-042"
EXPERIMENT_SLUG = "CNN_EXP042_WIDTH256_25K_RF1029"

SEED = 42

START_STEP = 10_000
STEPS = 25_000

BATCH_SIZE = 4
LOG_EVERY = 250

CHECKPOINT_STEPS = {
    15_000,
    20_000,
    25_000,
}

LEARNING_RATE = 1e-3
INTENSITY_LOSS_WEIGHT = 0.1

SOURCE_RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP040_WIDTH256_RF1029"
    / "run_001"
)

RESUME_CHECKPOINT = (
    SOURCE_RUN_DIR
    / "width256_final.pt"
)

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_CHECKPOINT = (
    RUN_DIR
    / "width256_25k_final.pt"
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

assert RESUME_CHECKPOINT.is_file(), (
    f"Не найден EXP040 @10k checkpoint: {RESUME_CHECKPOINT}"
)

print("Experiment:", EXPERIMENT_ID)
print("Resume:", RESUME_CHECKPOINT)
print("Start step:", START_STEP)
print("Final step:", STEPS)
print("Additional steps:", STEPS - START_STEP)
print("Channels:", EXPECTED_CHANNELS)
print("RF:", EXPECTED_RF)
print("LR:", LEARNING_RATE)
print("Batch:", BATCH_SIZE)
print("Artifacts:", RUN_DIR)


Experiment: CNN-EXP-042
Resume: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP040_WIDTH256_RF1029\run_001\width256_final.pt
Start step: 10000
Final step: 25000
Additional steps: 15000
Channels: 256
RF: 1029
LR: 0.001
Batch: 4
Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001


In [7]:
# ============================================================
# RESUME TRAINING: EXP040 10K -> EXP042 25K
# ============================================================

resume = torch.load(
    RESUME_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert int(resume["step"]) == START_STEP
assert int(resume["model_config"]["channels"]) == 256
assert int(resume["model_config"]["theoretical_rf"]) == 1029
assert int(resume["training_config"]["batch_size"]) == BATCH_SIZE

source_lr = float(
    resume["training_config"]["learning_rate"]
)

assert math.isclose(
    source_lr,
    LEARNING_RATE,
    rel_tol=0.0,
    abs_tol=1e-12,
), (
    f"EXP040 checkpoint LR={source_lr}, "
    f"но EXP042 ожидает LR={LEARNING_RATE}"
)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

generator.rng.bit_generator.state = copy.deepcopy(
    resume["generator_rng_state"]
)

model = (
    create_cnn_presence_intensity(channels=256)
    .to(device)
)

model.load_state_dict(
    resume["model_state_dict"]
)

assert model.backbone.stem.out_channels == EXPECTED_CHANNELS

optimizer = DirectMLAdam(
    model.parameters(),
    lr=LEARNING_RATE,
)

optimizer.load_state_dict(
    resume["optimizer_state_dict"]
)

for group in optimizer.param_groups:
    assert math.isclose(
        float(group["lr"]),
        LEARNING_RATE,
        rel_tol=0.0,
        abs_tol=1e-12,
    ), group["lr"]

torch.set_rng_state(
    resume["torch_rng_state"]
)


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def checkpoint_path(step):
    return (
        RUN_DIR
        / f"model_step_{step:05d}.pt"
    )


def save_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": EXPERIMENT_ID,
        "model_type": "presence_intensity",
        "architecture": "rf1029_width256",
        "step": int(step),

        "resumed_from_experiment": "CNN-EXP-040",
        "resumed_from_step": START_STEP,
        "resumed_from_checkpoint": str(
            RESUME_CHECKPOINT
        ),

        "model_state_dict": model_state_on_cpu(
            model
        ),

        "optimizer_state_dict": tree_to_cpu(
            optimizer.state_dict()
        ),

        "generator_rng_state": copy.deepcopy(
            generator.rng.bit_generator.state
        ),

        "torch_rng_state": torch.get_rng_state(),

        "model_config": {
            "input_channels": 5,
            "channels": 256,
            "output_channels": 2,
            "theoretical_rf": 1029,
        },

        "profile_config": {
            "target_length": profile_config.target_length,
            "context_flank": profile_config.context_flank,
            "positive_branch_fraction": (
                profile_config.positive_branch_fraction
            ),
        },

        "training_config": {
            "seed": SEED,
            "start_step": START_STEP,
            "steps": STEPS,
            "additional_steps": (
                STEPS - START_STEP
            ),
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "intensity_loss_weight": INTENSITY_LOSS_WEIGHT,
            "huber_delta": 1.0,
        },

        "seen_position_strand": int(
            seen_position_strand
        ),

        "elapsed_seconds": float(
            elapsed_seconds
        ),

        "history": list(history),
    }

    torch.save(
        checkpoint,
        path,
    )


history = list(
    resume.get(
        "history",
        [],
    )
)

assert history, (
    "EXP040 checkpoint does not contain training history."
)

last_record = history[-1]
assert int(last_record["step"]) == START_STEP

ema_presence = float(
    last_record["ema_presence"]
)

ema_intensity = float(
    last_record["ema_intensity"]
)

ema_total = float(
    last_record["ema_total"]
)

seen_position_strand = int(
    resume["seen_position_strand"]
)

previous_elapsed = float(
    resume["elapsed_seconds"]
)

del resume
gc.collect()

print("RESUME STATE")
print("step:", START_STEP)
print("EMA presence:", ema_presence)
print("EMA intensity:", ema_intensity)
print("EMA total:", ema_total)
print("seen position-strand:", seen_position_strand)
print("previous elapsed:", round(previous_elapsed, 2), "sec")
print("history records:", len(history))
print()

model.train()

continuation_started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(
        START_STEP + 1,
        STEPS + 1,
    ):
        batch = generator.sample_batch(
            batch_size=BATCH_SIZE
        )

        x_gpu = batch.x_both_strands.to(device)

        plus_target_gpu = (
            batch.plus_target.to(device)
        )

        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )

        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )

        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(
                plus_count
            )
            .to(device)
        )

        minus_intensity_target_reverse_gpu = (
            torch.log1p(
                minus_count_reverse
            )
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            model,
            x_gpu,
            batch_size=BATCH_SIZE,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = float(
            presence_loss.detach().cpu().item()
        )

        intensity_value = float(
            intensity_loss.detach().cpu().item()
        )

        total_value = float(
            total_loss.detach().cpu().item()
        )

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        optimizer.step()

        ema_presence = (
            0.99 * ema_presence
            + 0.01 * presence_value
        )

        ema_intensity = (
            0.99 * ema_intensity
            + 0.01 * intensity_value
        )

        ema_total = (
            0.99 * ema_total
            + 0.01 * total_value
        )

        seen_position_strand += int(
            batch.mask.sum()
        )

        continuation_elapsed = (
            time.perf_counter()
            - continuation_started_at
        )

        cumulative_elapsed = (
            previous_elapsed
            + continuation_elapsed
        )

        if (
            step % LOG_EVERY == 0
            or step == STEPS
        ):
            record = {
                "step": int(step),
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": float(ema_presence),
                "ema_intensity": float(ema_intensity),
                "ema_total": float(ema_total),
                "elapsed_seconds": float(cumulative_elapsed),
                "seen_position_strand": int(seen_position_strand),
                "positions_per_second": float(
                    seen_position_strand
                    / cumulative_elapsed
                ),
            }

            history.append(record)

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / cumulative_elapsed:,.0f}"
            )

        if step in CHECKPOINT_STEPS:
            save_checkpoint(
                checkpoint_path(step),
                step=step,
                elapsed_seconds=cumulative_elapsed,
                seen_position_strand=seen_position_strand,
                history=history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

continuation_elapsed = (
    time.perf_counter()
    - continuation_started_at
)

cumulative_elapsed = (
    previous_elapsed
    + continuation_elapsed
)

save_checkpoint(
    FINAL_CHECKPOINT,
    step=STEPS,
    elapsed_seconds=cumulative_elapsed,
    seen_position_strand=seen_position_strand,
    history=history,
)

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print(
    "Continuation time:",
    round(continuation_elapsed, 2),
    "sec",
)

print(
    "Cumulative time:",
    round(cumulative_elapsed, 2),
    "sec",
)

print(
    "Continuation time/step:",
    round(
        continuation_elapsed
        / (
            STEPS
            - START_STEP
        ),
        4,
    ),
    "sec",
)

print("Seen position-strand:", seen_position_strand)
print("Final EMA presence:", round(ema_presence, 6))
print("Final EMA intensity:", round(ema_intensity, 6))
print("Final EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", FINAL_CHECKPOINT)
print("History:", HISTORY_PATH)

assert not fallback_warnings
assert FINAL_CHECKPOINT.is_file()
assert HISTORY_PATH.is_file()


RESUME STATE
step: 10000
EMA presence: 0.3522626757218218
EMA intensity: 0.2567333698446849
EMA total: 0.37793601380894076
seen position-strand: 76146258
previous elapsed: 778.87 sec
history records: 41

step=10250 presence=0.7413 intensity=0.8262 total=0.8239 ema=(0.3762, 0.2524, 0.4015) positions/s=97,149
step=10500 presence=0.4904 intensity=0.0716 total=0.4975 ema=(0.3662, 0.2440, 0.3906) positions/s=96,651
step=10750 presence=0.4177 intensity=0.0217 total=0.4198 ema=(0.3661, 0.2621, 0.3923) positions/s=96,176
step=11000 presence=0.2722 intensity=0.7774 total=0.3500 ema=(0.3703, 0.2560, 0.3959) positions/s=95,721
step=11250 presence=0.4170 intensity=0.3745 total=0.4545 ema=(0.3783, 0.2593, 0.4042) positions/s=95,310
step=11500 presence=0.1965 intensity=0.4957 total=0.2461 ema=(0.3707, 0.2563, 0.3963) positions/s=94,920
step=11750 presence=0.5554 intensity=0.3469 total=0.5901 ema=(0.3520, 0.2453, 0.3765) positions/s=94,549
step=12000 presence=0.5455 intensity=0.3675 total=0.5822 ema=

# Сильная validation @25k

Тот же full-holdout protocol, что в EXP040 @10k.

Считаем:
- presence AP 5dp;
- presence Spearman raw / 5dp;
- intensity Spearman;
- per-contig;
- LOW_TP / HIGH_TP / HARD_FP / random TN;
- frozen feature probes;
- LOW_TP vs score-matched natural TN.


In [8]:
# ============================================================
# LOAD 10K CHECKPOINT
# ============================================================

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

validation_model = (
    create_cnn_presence_intensity(channels=256)
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

assert (
    validation_model
    .backbone
    .stem
    .out_channels
    == EXPECTED_CHANNELS
)

validation_model = validation_model.to(
    device
)

validation_model.eval()

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_tiles))
print(
    "Validation position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)

del checkpoint
gc.collect()


Checkpoint step: 25000
Validation tiles: 47743
Validation position-strand: 83807486
Validation positives: 56205


1070

In [9]:
# ============================================================
# METRIC HELPERS
# ============================================================

SUBMISSION_LEVELS = 100_001


def sigmoid_numpy(x):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    out = np.empty_like(
        x,
        dtype=np.float64,
    )

    positive = (
        x >= 0
    )

    out[
        positive
    ] = (
        1.0
        / (
            1.0
            + np.exp(
                -x[
                    positive
                ]
            )
        )
    )

    exp_x = np.exp(
        x[
            ~positive
        ]
    )

    out[
        ~positive
    ] = (
        exp_x
        / (
            1.0
            + exp_x
        )
    )

    return out.astype(
        np.float32
    )


def probability_bin_5dp(
    probabilities,
):
    return np.rint(
        np.clip(
            probabilities,
            0.0,
            1.0,
        )
        * 100_000
    ).astype(
        np.int32
    )


def update_histogram(
    hist,
    indices,
):
    values, counts = np.unique(
        indices,
        return_counts=True,
    )

    hist[
        values
    ] += counts.astype(
        np.int64
    )


def ap_from_histograms(
    positive_hist,
    total_hist,
):
    positives = int(
        positive_hist.sum()
    )

    if positives <= 0:
        return float("nan")

    pos_desc = (
        positive_hist[
            ::-1
        ]
    )

    total_desc = (
        total_hist[
            ::-1
        ]
    )

    tp = np.cumsum(
        pos_desc,
        dtype=np.int64,
    )

    predicted_positive = np.cumsum(
        total_desc,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.maximum(
            predicted_positive,
            1,
        )
    )

    return float(
        np.sum(
            (
                pos_desc
                / positives
            )
            * precision
        )
    )


def dense_rank_pearson(
    prediction,
    target,
):
    prediction = np.asarray(
        prediction
    )

    target = np.asarray(
        target
    )

    if len(prediction) < 2:
        return float("nan")

    pred_rank = rankdata(
        prediction,
        method="dense",
    )

    target_rank = rankdata(
        target,
        method="dense",
    )

    return float(
        pearsonr(
            pred_rank,
            target_rank,
        )[0]
    )


def new_segment_state():
    return {
        "total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_presence_raw": [],
        "positive_presence_5dp": [],
        "positive_intensity": [],
        "positive_count": [],
        "random_negative_score": [],
        "hard_negative_score": [],
    }


In [10]:
# ============================================================
# MEMORY-SAFE FULL HOLDOUT
# ============================================================

EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 100

RANDOM_NEG_PER_TILE_STRAND = 2
HARD_NEG_PER_TILE_STRAND = 2

rng = np.random.default_rng(
    20260924
)

segment_names = [
    "overall",
    *sorted(
        validation_tiles[
            "contig"
        ].astype(str).unique()
    ),
]

states = {
    segment: new_segment_state()
    for segment in segment_names
}

positions_seen = 0
started_eval = time.perf_counter()

validation_model.eval()

with torch.no_grad():

    for batch_no, start in enumerate(
        range(
            0,
            len(validation_tiles),
            EVAL_BATCH_SIZE,
        ),
        1,
    ):
        selected = (
            validation_tiles
            .iloc[
                start:
                start + EVAL_BATCH_SIZE
            ]
        )

        batch = (
            validation_generator
            .make_batch(
                selected
            )
        )

        B = len(selected)

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        (
            plus_presence_gpu,
            minus_presence_reverse_gpu,
            plus_intensity_gpu,
            minus_intensity_reverse_gpu,
        ) = forward_both_strands_multitask(
            validation_model,
            x_gpu,
            batch_size=B,
            context_flank=(
                profile_config
                .context_flank
            ),
            target_length=(
                profile_config
                .target_length
            ),
        )

        # GPU -> CPU immediately.
        plus_presence_logit = (
            plus_presence_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_presence_logit_reverse = (
            minus_presence_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        plus_intensity = (
            plus_intensity_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_intensity_reverse = (
            minus_intensity_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        del (
            x_gpu,
            plus_presence_gpu,
            minus_presence_reverse_gpu,
            plus_intensity_gpu,
            minus_intensity_reverse_gpu,
        )

        plus_probability = sigmoid_numpy(
            plus_presence_logit
        )

        minus_probability_reverse = sigmoid_numpy(
            minus_presence_logit_reverse
        )

        plus_count = (
            batch.count[
                :,
                0,
                :,
            ]
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        minus_count_reverse = (
            batch.count[
                :,
                1,
                :,
            ]
            .flip(-1)
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        plus_mask = (
            batch.mask[
                :,
                0,
                :,
            ]
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        minus_mask_reverse = (
            batch.mask[
                :,
                1,
                :,
            ]
            .flip(-1)
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        for local_i in range(B):
            contig = str(
                selected.iloc[
                    local_i
                ]["contig"]
            )

            for (
                probability,
                intensity,
                count,
                mask,
            ) in (
                (
                    plus_probability[
                        local_i
                    ],
                    plus_intensity[
                        local_i
                    ],
                    plus_count[
                        local_i
                    ],
                    plus_mask[
                        local_i
                    ],
                ),
                (
                    minus_probability_reverse[
                        local_i
                    ],
                    minus_intensity_reverse[
                        local_i
                    ],
                    minus_count_reverse[
                        local_i
                    ],
                    minus_mask_reverse[
                        local_i
                    ],
                ),
            ):
                valid_probability = (
                    probability[
                        mask
                    ]
                )

                valid_intensity = (
                    intensity[
                        mask
                    ]
                )

                valid_count = (
                    count[
                        mask
                    ]
                )

                positive = (
                    valid_count > 0
                )

                score_bin = (
                    probability_bin_5dp(
                        valid_probability
                    )
                )

                for segment in (
                    "overall",
                    contig,
                ):
                    state = states[
                        segment
                    ]

                    update_histogram(
                        state[
                            "total_hist"
                        ],
                        score_bin,
                    )

                    if positive.any():
                        update_histogram(
                            state[
                                "positive_hist"
                            ],
                            score_bin[
                                positive
                            ],
                        )

                        state[
                            "positive_presence_raw"
                        ].append(
                            valid_probability[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_presence_5dp"
                        ].append(
                            (
                                score_bin[
                                    positive
                                ]
                                / 100_000.0
                            ).astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_intensity"
                        ].append(
                            valid_intensity[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_count"
                        ].append(
                            valid_count[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                negative_index = np.flatnonzero(
                    ~positive
                )

                if len(negative_index):

                    n_random = min(
                        RANDOM_NEG_PER_TILE_STRAND,
                        len(
                            negative_index
                        ),
                    )

                    random_index = rng.choice(
                        negative_index,
                        size=n_random,
                        replace=False,
                    )

                    n_hard = min(
                        HARD_NEG_PER_TILE_STRAND,
                        len(
                            negative_index
                        ),
                    )

                    negative_score = (
                        valid_probability[
                            negative_index
                        ]
                    )

                    if (
                        len(
                            negative_index
                        )
                        <= n_hard
                    ):
                        hard_index = (
                            negative_index
                        )
                    else:
                        local_top = np.argpartition(
                            negative_score,
                            -n_hard,
                        )[
                            -n_hard:
                        ]

                        hard_index = (
                            negative_index[
                                local_top
                            ]
                        )

                    for segment in (
                        "overall",
                        contig,
                    ):
                        states[
                            segment
                        ][
                            "random_negative_score"
                        ].append(
                            valid_probability[
                                random_index
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        states[
                            segment
                        ][
                            "hard_negative_score"
                        ].append(
                            valid_probability[
                                hard_index
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                positions_seen += int(
                    mask.sum()
                )

        del (
            batch,
            selected,
            plus_presence_logit,
            minus_presence_logit_reverse,
            plus_intensity,
            minus_intensity_reverse,
            plus_probability,
            minus_probability_reverse,
            plus_count,
            minus_count_reverse,
            plus_mask,
            minus_mask_reverse,
        )

        if batch_no % 25 == 0:
            gc.collect()

        if (
            batch_no == 1
            or batch_no % PROGRESS_EVERY == 0
            or start + B >= len(
                validation_tiles
            )
        ):
            elapsed = (
                time.perf_counter()
                - started_eval
            )

            print(
                f"tiles={start + B:>6}"
                f"/{len(validation_tiles)} "
                f"positions={positions_seen:>12,} "
                f"positions/s="
                f"{positions_seen / elapsed:,.0f}"
            )

print()
print("FULL HOLDOUT FINISHED")


tiles=     1/47743 positions=         210 positions/s=19,037
tiles=   100/47743 positions=     125,828 positions/s=81,728
tiles=   200/47743 positions=     269,398 positions/s=90,333
tiles=   300/47743 positions=     403,094 positions/s=91,268
tiles=   400/47743 positions=     552,594 positions/s=94,251
tiles=   500/47743 positions=     708,204 positions/s=96,431
tiles=   600/47743 positions=     871,458 positions/s=99,245
tiles=   700/47743 positions=   1,020,090 positions/s=99,809
tiles=   800/47743 positions=   1,185,336 positions/s=101,287
tiles=   900/47743 positions=   1,346,300 positions/s=102,449
tiles=  1000/47743 positions=   1,519,462 positions/s=104,200
tiles=  1100/47743 positions=   1,700,978 positions/s=106,183
tiles=  1200/47743 positions=   1,853,300 positions/s=106,201
tiles=  1300/47743 positions=   2,017,896 positions/s=106,842
tiles=  1400/47743 positions=   2,193,246 positions/s=107,915
tiles=  1500/47743 positions=   2,381,142 positions/s=109,422
tiles=  1600/477

In [11]:
# ============================================================
# OVERALL + PER-CONTIG METRICS
# ============================================================

metric_rows = []

cohort_cache = {}

for segment, state in states.items():

    positive_raw = np.concatenate(
        state[
            "positive_presence_raw"
        ]
    )

    positive_5dp = np.concatenate(
        state[
            "positive_presence_5dp"
        ]
    )

    positive_intensity = np.concatenate(
        state[
            "positive_intensity"
        ]
    )

    positive_count = np.concatenate(
        state[
            "positive_count"
        ]
    )

    random_negative = np.concatenate(
        state[
            "random_negative_score"
        ]
    )

    hard_negative_candidates = np.concatenate(
        state[
            "hard_negative_score"
        ]
    )

    # Candidate pool contains top negatives from every tile.
    # Keep only its global highest tail.
    hard_keep = min(
        20_000,
        len(
            hard_negative_candidates
        ),
    )

    if (
        len(
            hard_negative_candidates
        )
        > hard_keep
    ):
        hard_idx = np.argpartition(
            hard_negative_candidates,
            -hard_keep,
        )[
            -hard_keep:
        ]

        hard_negative = (
            hard_negative_candidates[
                hard_idx
            ]
        )
    else:
        hard_negative = (
            hard_negative_candidates
        )

    presence_ap_5dp = (
        ap_from_histograms(
            state[
                "positive_hist"
            ],
            state[
                "total_hist"
            ],
        )
    )

    presence_spearman_raw = (
        dense_rank_pearson(
            positive_raw,
            positive_count,
        )
    )

    presence_spearman_5dp = (
        dense_rank_pearson(
            positive_5dp,
            positive_count,
        )
    )

    intensity_spearman = (
        dense_rank_pearson(
            positive_intensity,
            positive_count,
        )
    )

    metric_rows.append({
        "segment": segment,
        "positive_n": int(
            len(
                positive_count
            )
        ),
        "presence_AP_5dp": float(
            presence_ap_5dp
        ),
        "presence_EPIC_Spearman_raw": float(
            presence_spearman_raw
        ),
        "presence_EPIC_Spearman_5dp": float(
            presence_spearman_5dp
        ),
        "intensity_EPIC_Spearman_raw": float(
            intensity_spearman
        ),
    })

    cohort_cache[
        segment
    ] = {
        "positive_score": (
            positive_raw
        ),
        "positive_count": (
            positive_count
        ),
        "random_negative_score": (
            random_negative
        ),
        "hard_negative_score": (
            hard_negative
        ),
    }

metrics = pd.DataFrame(
    metric_rows
).sort_values(
    by="segment"
).reset_index(
    drop=True
)

METRICS_PATH = (
    RUN_DIR
    / "strong_validation_metrics.csv"
)

metrics.to_csv(
    METRICS_PATH,
    index=False,
)

display(metrics)

print("Saved:", METRICS_PATH)


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.075334,0.200920,0.170139,0.222120
1,NC_064041.1,13984,0.068389,0.235345,0.198931,0.249091
2,NC_064042.1,18692,0.082411,0.225596,0.185773,0.249896
3,overall,56205,0.075671,0.187675,0.141133,0.208023


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\strong_validation_metrics.csv


## Ошибки: TP, LOW_TP, HARD_FP, TN

Это диагностика вопроса:

> **Что изменилось именно после 128 → 256 channels?**

Для overall:

```text
HIGH_TP = верхние 25% baseline score среди positives
LOW_TP  = нижние 25% baseline score среди positives

HARD_FP = самые высоко оценённые negative candidates
RANDOM_TN = случайная natural-background выборка negatives
```

Смотрим:
- вырос ли AP;
- стабилен ли прирост по contigs;
- изменился ли LOW_TP;
- изменился ли HARD_FP;
- появился ли в 256-D representation новый полезный signal.


In [12]:
# ============================================================
# ERROR-POPULATION STUDY
# ============================================================

overall = cohort_cache[
    "overall"
]

positive_score = overall[
    "positive_score"
]

positive_count = overall[
    "positive_count"
]

random_tn = overall[
    "random_negative_score"
]

hard_fp = overall[
    "hard_negative_score"
]

q25 = float(
    np.quantile(
        positive_score,
        0.25,
    )
)

q75 = float(
    np.quantile(
        positive_score,
        0.75,
    )
)

low_tp = (
    positive_score[
        positive_score <= q25
    ]
)

high_tp = (
    positive_score[
        positive_score >= q75
    ]
)


def describe_scores(
    name,
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "cohort": name,
        "n": int(
            len(values)
        ),
        "mean": float(
            values.mean()
        ),
        "p10": float(
            np.quantile(
                values,
                0.10,
            )
        ),
        "p50": float(
            np.quantile(
                values,
                0.50,
            )
        ),
        "p90": float(
            np.quantile(
                values,
                0.90,
            )
        ),
        "p99": float(
            np.quantile(
                values,
                0.99,
            )
        ),
    }


cohort_summary = pd.DataFrame([
    describe_scores(
        "HIGH_TP",
        high_tp,
    ),
    describe_scores(
        "LOW_TP",
        low_tp,
    ),
    describe_scores(
        "ALL_TP",
        positive_score,
    ),
    describe_scores(
        "HARD_FP",
        hard_fp,
    ),
    describe_scores(
        "RANDOM_TN",
        random_tn,
    ),
])

COHORT_PATH = (
    RUN_DIR
    / "error_population_score_summary.csv"
)

cohort_summary.to_csv(
    COHORT_PATH,
    index=False,
)

display(cohort_summary)

print("Positive q25:", q25)
print("Positive q75:", q75)
print("Saved:", COHORT_PATH)


,cohort,n,mean,p10,p50,p90,p99
0,HIGH_TP,14052,0.997865,0.994358,0.998680,0.999930,0.999997
1,LOW_TP,14052,0.397542,0.072889,0.419213,0.684896,0.725514
2,ALL_TP,56205,0.808204,0.334466,0.944784,0.999186,0.999982
3,HARD_FP,20000,0.998512,0.997076,0.998581,0.999814,0.999988
4,RANDOM_TN,190962,0.177844,0.002664,0.053718,0.601947,0.959828


Positive q25: 0.7299213409423828
Positive q75: 0.9925704598426819
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\error_population_score_summary.csv


In [13]:
# ============================================================
# POSITIVE STRENGTH STUDY
# ============================================================

strength_group = np.select(
    [
        positive_count == 1,
        positive_count == 2,
        (
            positive_count >= 3
        )
        & (
            positive_count <= 4
        ),
        positive_count >= 5,
    ],
    [
        "reads_1",
        "reads_2",
        "reads_3_4",
        "reads_5_plus",
    ],
    default="other",
)

strength_rows = []

for group in (
    "reads_1",
    "reads_2",
    "reads_3_4",
    "reads_5_plus",
):
    values = (
        positive_score[
            strength_group == group
        ]
    )

    if len(values) == 0:
        continue

    row = describe_scores(
        group,
        values,
    )

    strength_rows.append(
        row
    )

strength_summary = pd.DataFrame(
    strength_rows
)

STRENGTH_PATH = (
    RUN_DIR
    / "positive_strength_score_summary.csv"
)

strength_summary.to_csv(
    STRENGTH_PATH,
    index=False,
)

display(strength_summary)

print("Saved:", STRENGTH_PATH)


,cohort,n,mean,p10,p50,p90,p99
0,reads_1,29982,0.758349,0.246910,0.894878,0.996431,0.999912
1,reads_2,10496,0.804645,0.339312,0.938790,0.998669,0.999962
2,reads_3_4,6529,0.854805,0.467620,0.970652,0.999416,0.999982
3,reads_5_plus,9198,0.941697,0.846221,0.995627,0.999912,0.999997


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\positive_strength_score_summary.csv


# OPTIONAL — исследуем WIDTH256 representation после 25k

Ключевые сравнения:

```text
WIDTH256 @10k → WIDTH256 @25k
WIDTH128 @25k → WIDTH256 @25k
```

Особенно интересен LOW_TP score-matched probe.

Если 256@25k резко улучшит AP относительно 256@10k, то прежний вывод
про "WIDTH256 хуже" был следствием недостаточного training budget.

Если 256@25k обгонит 128@25k — capacity scaling продолжается.

Если 256@25k всё ещё хуже 128@25k — WIDTH128 остаётся лучшей шириной
при одинаковом 25k budget.


In [14]:
# ============================================================
# OPTIONAL FEATURE EXTRACTION
# ============================================================

RUN_FEATURE_PROBE = True

DIAG_TILES_PER_CONTIG = 1000
RANDOM_NEG_FEATURES_PER_TILE_STRAND = 2
HARD_NEG_FEATURES_PER_TILE_STRAND = 2

FEATURES_NPZ = (
    RUN_DIR
    / "width256_25k_feature_probe_samples.npz"
)

if RUN_FEATURE_PROBE:

    diag_parts = []

    for contig, frame in (
        validation_tiles
        .groupby(
            "contig",
            sort=True,
        )
    ):
        n = min(
            DIAG_TILES_PER_CONTIG,
            len(frame),
        )

        positions = np.linspace(
            0,
            len(frame) - 1,
            n,
            dtype=int,
        )

        positions = np.unique(
            positions
        )

        diag_parts.append(
            frame.iloc[
                positions
            ]
        )

    diag_tiles = (
        pd.concat(
            diag_parts,
            axis=0,
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "Diagnostic tiles:",
        len(diag_tiles),
    )

    rng_probe = np.random.default_rng(
        20260925
    )

    tp_X = []
    tp_score = []
    tp_count = []
    tp_contig = []

    tn_X = []
    tn_score = []
    tn_contig = []

    hard_X = []
    hard_score = []
    hard_contig = []

    validation_model.eval()

    started_probe = time.perf_counter()

    with torch.no_grad():

        for row_i in range(
            len(diag_tiles)
        ):
            selected = (
                diag_tiles
                .iloc[
                    row_i:
                    row_i + 1
                ]
            )

            batch = (
                validation_generator
                .make_batch(
                    selected
                )
            )

            contig = str(
                selected.iloc[
                    0
                ]["contig"]
            )

            x_gpu = (
                batch
                .x_both_strands
                .to(device)
            )

            feature_gpu = (
                validation_model
                .backbone(
                    x_gpu
                )
            )

            output_gpu = (
                validation_model
                .profile_head(
                    feature_gpu
                )
            )

            # CPU first.
            feature = (
                feature_gpu
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            output = (
                output_gpu
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            del (
                x_gpu,
                feature_gpu,
                output_gpu,
            )

            c = (
                profile_config
                .context_flank
            )

            L = (
                profile_config
                .target_length
            )

            feature = (
                feature[
                    :,
                    :,
                    c:
                    c + L
                ]
            )

            output = (
                output[
                    :,
                    :,
                    c:
                    c + L
                ]
            )

            # B=1:
            # index 0 = plus orientation
            # index 1 = reverse-complement orientation.
            plus_count = (
                batch.count[
                    0,
                    0,
                    :,
                ]
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            minus_count_reverse = (
                batch.count[
                    0,
                    1,
                    :,
                ]
                .flip(-1)
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            plus_mask = (
                batch.mask[
                    0,
                    0,
                    :,
                ]
                .numpy()
                .astype(
                    bool,
                    copy=False,
                )
            )

            minus_mask_reverse = (
                batch.mask[
                    0,
                    1,
                    :,
                ]
                .flip(-1)
                .numpy()
                .astype(
                    bool,
                    copy=False,
                )
            )

            for orient_i, (
                count,
                mask,
            ) in enumerate(
                (
                    (
                        plus_count,
                        plus_mask,
                    ),
                    (
                        minus_count_reverse,
                        minus_mask_reverse,
                    ),
                )
            ):
                feat = (
                    feature[
                        orient_i
                    ]
                    .T
                )

                probability = (
                    sigmoid_numpy(
                        output[
                            orient_i,
                            0,
                            :,
                        ]
                    )
                )

                valid_index = np.flatnonzero(
                    mask
                )

                pos_index = valid_index[
                    count[
                        valid_index
                    ] > 0
                ]

                neg_index = valid_index[
                    count[
                        valid_index
                    ] == 0
                ]

                if len(pos_index):
                    tp_X.append(
                        feat[
                            pos_index
                        ].copy()
                    )

                    tp_score.append(
                        probability[
                            pos_index
                        ].copy()
                    )

                    tp_count.append(
                        count[
                            pos_index
                        ].copy()
                    )

                    tp_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            pos_index
                        )
                    )

                if len(neg_index):
                    n_random = min(
                        RANDOM_NEG_FEATURES_PER_TILE_STRAND,
                        len(
                            neg_index
                        ),
                    )

                    random_index = rng_probe.choice(
                        neg_index,
                        size=n_random,
                        replace=False,
                    )

                    n_hard = min(
                        HARD_NEG_FEATURES_PER_TILE_STRAND,
                        len(
                            neg_index
                        ),
                    )

                    neg_probability = (
                        probability[
                            neg_index
                        ]
                    )

                    if len(
                        neg_index
                    ) <= n_hard:
                        hard_index = (
                            neg_index
                        )
                    else:
                        local_top = np.argpartition(
                            neg_probability,
                            -n_hard,
                        )[
                            -n_hard:
                        ]

                        hard_index = (
                            neg_index[
                                local_top
                            ]
                        )

                    tn_X.append(
                        feat[
                            random_index
                        ].copy()
                    )

                    tn_score.append(
                        probability[
                            random_index
                        ].copy()
                    )

                    tn_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            random_index
                        )
                    )

                    hard_X.append(
                        feat[
                            hard_index
                        ].copy()
                    )

                    hard_score.append(
                        probability[
                            hard_index
                        ].copy()
                    )

                    hard_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            hard_index
                        )
                    )

            del (
                batch,
                selected,
                feature,
                output,
            )

            if (
                row_i == 0
                or (
                    row_i + 1
                ) % 250 == 0
                or row_i + 1
                == len(
                    diag_tiles
                )
            ):
                elapsed = (
                    time.perf_counter()
                    - started_probe
                )

                print(
                    f"probe tiles="
                    f"{row_i + 1:>5}"
                    f"/{len(diag_tiles)} "
                    f"elapsed="
                    f"{elapsed / 60:.1f} min"
                )

            if (
                row_i + 1
            ) % 25 == 0:
                gc.collect()

    tp_X = np.concatenate(
        tp_X,
        axis=0,
    )

    tp_score = np.concatenate(
        tp_score
    )

    tp_count = np.concatenate(
        tp_count
    )

    tp_contig = np.asarray(
        tp_contig
    )

    tn_X = np.concatenate(
        tn_X,
        axis=0,
    )

    tn_score = np.concatenate(
        tn_score
    )

    tn_contig = np.asarray(
        tn_contig
    )

    hard_X = np.concatenate(
        hard_X,
        axis=0,
    )

    hard_score = np.concatenate(
        hard_score
    )

    hard_contig = np.asarray(
        hard_contig
    )

    np.savez_compressed(
        FEATURES_NPZ,
        tp_X=tp_X,
        tp_score=tp_score,
        tp_count=tp_count,
        tp_contig=tp_contig,
        random_tn_X=tn_X,
        random_tn_score=tn_score,
        random_tn_contig=tn_contig,
        hard_fp_X=hard_X,
        hard_fp_score=hard_score,
        hard_fp_contig=hard_contig,
    )

    print()
    print("TP:", tp_X.shape)
    print("Random TN:", tn_X.shape)
    print("Hard FP:", hard_X.shape)
    print("Saved:", FEATURES_NPZ)

else:
    print("Feature probe disabled.")


Diagnostic tiles: 3000
probe tiles=    1/3000 elapsed=0.0 min
probe tiles=  250/3000 elapsed=0.1 min
probe tiles=  500/3000 elapsed=0.1 min
probe tiles=  750/3000 elapsed=0.2 min
probe tiles= 1000/3000 elapsed=0.3 min
probe tiles= 1250/3000 elapsed=0.3 min
probe tiles= 1500/3000 elapsed=0.4 min
probe tiles= 1750/3000 elapsed=0.5 min
probe tiles= 2000/3000 elapsed=0.5 min
probe tiles= 2250/3000 elapsed=0.6 min
probe tiles= 2500/3000 elapsed=0.7 min
probe tiles= 2750/3000 elapsed=0.7 min
probe tiles= 3000/3000 elapsed=0.8 min

TP: (3483, 256)
Random TN: (11998, 256)
Hard FP: (11998, 256)
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width256_25k_feature_probe_samples.npz


In [15]:
# ============================================================
# OPTIONAL CROSS-CONTIG LINEAR PROBE
# ============================================================

if RUN_FEATURE_PROBE:

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    data = np.load(
        FEATURES_NPZ,
        allow_pickle=False,
    )

    tp_X = data[
        "tp_X"
    ]

    tp_score = data[
        "tp_score"
    ]

    tp_contig = data[
        "tp_contig"
    ]

    random_tn_X = data[
        "random_tn_X"
    ]

    random_tn_score = data[
        "random_tn_score"
    ]

    random_tn_contig = data[
        "random_tn_contig"
    ]

    hard_fp_X = data[
        "hard_fp_X"
    ]

    hard_fp_score = data[
        "hard_fp_score"
    ]

    hard_fp_contig = data[
        "hard_fp_contig"
    ]


    def cross_contig_probe(
        positive_X,
        positive_score,
        positive_contig,
        negative_X,
        negative_score,
        negative_contig,
        task_name,
    ):
        rows = []

        contigs = sorted(
            set(
                positive_contig.tolist()
            )
            & set(
                negative_contig.tolist()
            )
        )

        for heldout in contigs:

            pos_train = (
                positive_contig
                != heldout
            )

            neg_train = (
                negative_contig
                != heldout
            )

            pos_test = (
                positive_contig
                == heldout
            )

            neg_test = (
                negative_contig
                == heldout
            )

            X_train = np.concatenate(
                (
                    positive_X[
                        pos_train
                    ],
                    negative_X[
                        neg_train
                    ],
                ),
                axis=0,
            )

            y_train = np.concatenate(
                (
                    np.ones(
                        pos_train.sum(),
                        dtype=np.int8,
                    ),
                    np.zeros(
                        neg_train.sum(),
                        dtype=np.int8,
                    ),
                )
            )

            X_test = np.concatenate(
                (
                    positive_X[
                        pos_test
                    ],
                    negative_X[
                        neg_test
                    ],
                ),
                axis=0,
            )

            y_test = np.concatenate(
                (
                    np.ones(
                        pos_test.sum(),
                        dtype=np.int8,
                    ),
                    np.zeros(
                        neg_test.sum(),
                        dtype=np.int8,
                    ),
                )
            )

            head_score = np.concatenate(
                (
                    positive_score[
                        pos_test
                    ],
                    negative_score[
                        neg_test
                    ],
                )
            )

            probe = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="lbfgs",
                ),
            )

            probe.fit(
                X_train,
                y_train,
            )

            probe_score = (
                probe.predict_proba(
                    X_test
                )[
                    :,
                    1
                ]
            )

            rows.append({
                "task": task_name,
                "heldout_contig": heldout,
                "n_test": int(
                    len(
                        y_test
                    )
                ),
                "presence_head_ROC_AUC": float(
                    roc_auc_score(
                        y_test,
                        head_score,
                    )
                ),
                "linear_probe_ROC_AUC": float(
                    roc_auc_score(
                        y_test,
                        probe_score,
                    )
                ),
            })

        return pd.DataFrame(
            rows
        )


    probe_random = cross_contig_probe(
        tp_X,
        tp_score,
        tp_contig,
        random_tn_X,
        random_tn_score,
        random_tn_contig,
        "TP_vs_RANDOM_TN",
    )

    probe_hard = cross_contig_probe(
        tp_X,
        tp_score,
        tp_contig,
        hard_fp_X,
        hard_fp_score,
        hard_fp_contig,
        "TP_vs_HARD_FP",
    )

    probe_results = pd.concat(
        (
            probe_random,
            probe_hard,
        ),
        ignore_index=True,
    )

    PROBE_PATH = (
        RUN_DIR
        / "width256_25k_cross_contig_feature_probe.csv"
    )

    probe_results.to_csv(
        PROBE_PATH,
        index=False,
    )

    display(probe_results)

    display(
        probe_results
        .groupby(
            "task",
            as_index=False,
        )[
            [
                "presence_head_ROC_AUC",
                "linear_probe_ROC_AUC",
            ]
        ]
        .mean()
    )

    print("Saved:", PROBE_PATH)

else:
    print("Feature probe disabled.")


,task,heldout_contig,n_test,presence_head_ROC_AUC,linear_probe_ROC_AUC
0,TP_vs_RANDOM_TN,NC_064034.1,5115,0.930652,0.920333
1,TP_vs_RANDOM_TN,NC_064041.1,5035,0.949670,0.943522
2,TP_vs_RANDOM_TN,NC_064042.1,5331,0.931434,0.920406
3,TP_vs_HARD_FP,NC_064034.1,5115,0.418058,0.827539
4,TP_vs_HARD_FP,NC_064041.1,5035,0.517536,0.829352
5,TP_vs_HARD_FP,NC_064042.1,5331,0.404278,0.841467


,task,presence_head_ROC_AUC,linear_probe_ROC_AUC
0,TP_vs_HARD_FP,0.446624,0.832786
1,TP_vs_RANDOM_TN,0.937252,0.928087


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width256_25k_cross_contig_feature_probe.csv


# LOW_TP vs score-matched natural TN — residual-information probe

Это ключевой representation test после WIDTH256.

Мы **не обучаем residual correction** и не меняем CNN.

Для каждого held-out validation contig:

```text
train:
    LOW_TP из остальных validation contigs
    vs score-matched random TN

test:
    LOW_TP held-out contig
    vs score-matched random TN held-out contig
```

LOW_TP = нижний квартиль TP по baseline presence score.

TN подбираются из тех же score ranges. Поэтому:

```text
baseline-score AUC ≈ 0.5
linear 256-D probe AUC > 0.5
```

означает, что representation содержит дополнительную информацию,
которой scalar presence score не использует.

Если доступен EXP036 `width64_feature_probe_samples.npz`,
тем же кодом считаем WIDTH64 для прямого сравнения.


In [16]:
# ============================================================
# LOW_TP vs SCORE-MATCHED RANDOM TN PROBE
# ============================================================

LOWTP_PROBE_PATH = (
    RUN_DIR
    / "width256_25k_lowtp_scorematched_probe.csv"
)

WIDTH_COMPARISON_PATH = (
    RUN_DIR
    / "width_training_scaling_lowtp_probe.csv"
)


def score_matched_negative_indices(
    positive_scores,
    negative_scores,
    *,
    seed,
    n_bins=10,
):
    rng = np.random.default_rng(seed)

    positive_scores = np.asarray(
        positive_scores,
        dtype=np.float64,
    )

    negative_scores = np.asarray(
        negative_scores,
        dtype=np.float64,
    )

    edges = np.unique(
        np.quantile(
            positive_scores,
            np.linspace(
                0.0,
                1.0,
                n_bins + 1,
            ),
        )
    )

    if len(edges) < 3:
        lo = float(
            positive_scores.min()
        )
        hi = float(
            positive_scores.max()
        )

        if hi <= lo:
            hi = lo + 1e-6

        edges = np.linspace(
            lo,
            hi,
            3,
        )

    chosen_parts = []

    for bin_i in range(
        len(edges) - 1
    ):
        left = edges[bin_i]
        right = edges[bin_i + 1]

        if bin_i == len(edges) - 2:
            pos_mask = (
                (positive_scores >= left)
                & (positive_scores <= right)
            )
            neg_mask = (
                (negative_scores >= left)
                & (negative_scores <= right)
            )
        else:
            pos_mask = (
                (positive_scores >= left)
                & (positive_scores < right)
            )
            neg_mask = (
                (negative_scores >= left)
                & (negative_scores < right)
            )

        n_pos = int(
            pos_mask.sum()
        )

        if n_pos == 0:
            continue

        candidates = np.flatnonzero(
            neg_mask
        )

        if len(candidates) == 0:
            center = 0.5 * (
                left + right
            )

            distance = np.abs(
                negative_scores
                - center
            )

            fallback_n = min(
                max(
                    n_pos,
                    16,
                ),
                len(
                    negative_scores
                ),
            )

            candidates = np.argpartition(
                distance,
                fallback_n - 1,
            )[
                :fallback_n
            ]

        selected = rng.choice(
            candidates,
            size=n_pos,
            replace=(
                n_pos
                > len(candidates)
            ),
        )

        chosen_parts.append(
            selected
        )

    if not chosen_parts:
        raise RuntimeError(
            "No score-matched TN candidates."
        )

    return np.concatenate(
        chosen_parts
    )


def lowtp_scorematched_probe_from_npz(
    npz_path,
    *,
    representation_name,
):
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    data = np.load(
        npz_path,
        allow_pickle=False,
    )

    X_pos = data["tp_X"]
    score_pos = data["tp_score"]
    contig_pos = data["tp_contig"]

    X_neg = data["random_tn_X"]
    score_neg = data["random_tn_score"]
    contig_neg = data["random_tn_contig"]

    rows = []

    contigs = sorted(
        set(
            contig_pos.tolist()
        )
        & set(
            contig_neg.tolist()
        )
    )

    for fold_i, heldout in enumerate(
        contigs,
        1,
    ):
        train_pos_mask = (
            contig_pos != heldout
        )
        test_pos_mask = (
            contig_pos == heldout
        )

        train_neg_mask = (
            contig_neg != heldout
        )
        test_neg_mask = (
            contig_neg == heldout
        )

        # LOW_TP is defined within train/test separately by baseline score.
        train_cutoff = float(
            np.quantile(
                score_pos[
                    train_pos_mask
                ],
                0.25,
            )
        )

        test_cutoff = float(
            np.quantile(
                score_pos[
                    test_pos_mask
                ],
                0.25,
            )
        )

        train_low_mask = (
            train_pos_mask
            & (
                score_pos
                <= train_cutoff
            )
        )

        test_low_mask = (
            test_pos_mask
            & (
                score_pos
                <= test_cutoff
            )
        )

        train_neg_idx_all = np.flatnonzero(
            train_neg_mask
        )

        test_neg_idx_all = np.flatnonzero(
            test_neg_mask
        )

        train_match_local = (
            score_matched_negative_indices(
                score_pos[
                    train_low_mask
                ],
                score_neg[
                    train_neg_mask
                ],
                seed=20260925 + fold_i,
            )
        )

        test_match_local = (
            score_matched_negative_indices(
                score_pos[
                    test_low_mask
                ],
                score_neg[
                    test_neg_mask
                ],
                seed=20261925 + fold_i,
            )
        )

        train_neg_idx = (
            train_neg_idx_all[
                train_match_local
            ]
        )

        test_neg_idx = (
            test_neg_idx_all[
                test_match_local
            ]
        )

        X_train = np.concatenate(
            (
                X_pos[
                    train_low_mask
                ],
                X_neg[
                    train_neg_idx
                ],
            ),
            axis=0,
        )

        y_train = np.concatenate(
            (
                np.ones(
                    int(
                        train_low_mask.sum()
                    ),
                    dtype=np.int8,
                ),
                np.zeros(
                    len(
                        train_neg_idx
                    ),
                    dtype=np.int8,
                ),
            )
        )

        X_test = np.concatenate(
            (
                X_pos[
                    test_low_mask
                ],
                X_neg[
                    test_neg_idx
                ],
            ),
            axis=0,
        )

        y_test = np.concatenate(
            (
                np.ones(
                    int(
                        test_low_mask.sum()
                    ),
                    dtype=np.int8,
                ),
                np.zeros(
                    len(
                        test_neg_idx
                    ),
                    dtype=np.int8,
                ),
            )
        )

        baseline_score = np.concatenate(
            (
                score_pos[
                    test_low_mask
                ],
                score_neg[
                    test_neg_idx
                ],
            )
        )

        probe = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                solver="lbfgs",
            ),
        )

        probe.fit(
            X_train,
            y_train,
        )

        probe_score = (
            probe
            .predict_proba(
                X_test
            )[
                :,
                1
            ]
        )

        rows.append({
            "representation": representation_name,
            "heldout_contig": heldout,
            "n_low_tp_test": int(
                test_low_mask.sum()
            ),
            "n_scorematched_tn_test": int(
                len(
                    test_neg_idx
                )
            ),
            "baseline_score_ROC_AUC": float(
                roc_auc_score(
                    y_test,
                    baseline_score,
                )
            ),
            "linear_probe_ROC_AUC": float(
                roc_auc_score(
                    y_test,
                    probe_score,
                )
            ),
        })

    return pd.DataFrame(
        rows
    )




if RUN_FEATURE_PROBE:

    width256_25k_lowtp_probe = (
        lowtp_scorematched_probe_from_npz(
            FEATURES_NPZ,
            representation_name="WIDTH256_25K",
        )
    )

    width256_25k_lowtp_probe.to_csv(
        LOWTP_PROBE_PATH,
        index=False,
    )

    print("WIDTH256 @25k LOW_TP score-matched probe:")
    display(width256_25k_lowtp_probe)

    comparison_parts = []

    source_feature_files = [
        (
            "WIDTH64_10K",
            PROJECT_ROOT
            / "artifacts"
            / "experiments"
            / "CNN_EXP036_WIDTH64_RF1029"
            / "run_001"
            / "width64_feature_probe_samples.npz",
        ),
        (
            "WIDTH128_10K",
            PROJECT_ROOT
            / "artifacts"
            / "experiments"
            / "CNN_EXP039_WIDTH128_RF1029"
            / "run_001"
            / "width128_feature_probe_samples.npz",
        ),
        (
            "WIDTH128_25K",
            PROJECT_ROOT
            / "artifacts"
            / "experiments"
            / "CNN_EXP041_WIDTH128_25K_RF1029"
            / "run_001"
            / "width128_25k_feature_probe_samples.npz",
        ),
        (
            "WIDTH256_10K",
            PROJECT_ROOT
            / "artifacts"
            / "experiments"
            / "CNN_EXP040_WIDTH256_RF1029"
            / "run_001"
            / "width256_feature_probe_samples.npz",
        ),
    ]

    for representation_name, feature_path in source_feature_files:
        if feature_path.is_file():
            comparison_parts.append(
                lowtp_scorematched_probe_from_npz(
                    feature_path,
                    representation_name=representation_name,
                )
            )
        else:
            print(
                representation_name,
                "feature NPZ not found:",
                feature_path,
            )

    comparison_parts.append(
        width256_25k_lowtp_probe
    )

    training_compare = pd.concat(
        comparison_parts,
        ignore_index=True,
    )

    training_compare.to_csv(
        WIDTH_COMPARISON_PATH,
        index=False,
    )

    print()
    print("Width / training-budget representation comparison:")
    display(training_compare)

    print()
    print("Mean probe AUC:")
    display(
        training_compare
        .groupby(
            "representation",
            as_index=False,
        )[
            [
                "baseline_score_ROC_AUC",
                "linear_probe_ROC_AUC",
            ]
        ]
        .mean()
    )

    print("Saved:", LOWTP_PROBE_PATH)
    print("Saved:", WIDTH_COMPARISON_PATH)

else:
    print("RUN_FEATURE_PROBE=False; LOW_TP probe skipped.")


WIDTH256 @25k LOW_TP score-matched probe:


,representation,heldout_contig,n_low_tp_test,n_scorematched_tn_test,baseline_score_ROC_AUC,linear_probe_ROC_AUC
0,WIDTH256_25K,NC_064034.1,279,279,0.507136,0.478450
1,WIDTH256_25K,NC_064041.1,259,259,0.504748,0.552958
2,WIDTH256_25K,NC_064042.1,334,334,0.500341,0.505791



Width / training-budget representation comparison:


,representation,heldout_contig,n_low_tp_test,n_scorematched_tn_test,baseline_score_ROC_AUC,linear_probe_ROC_AUC
0,WIDTH64_10K,NC_064034.1,279,279,0.502023,0.520985
1,WIDTH64_10K,NC_064041.1,259,259,0.506656,0.519372
2,WIDTH64_10K,NC_064042.1,334,334,0.503729,0.502591
3,WIDTH128_10K,NC_064034.1,279,279,0.500430,0.586233
4,WIDTH128_10K,NC_064041.1,259,259,0.507238,0.597174
5,WIDTH128_10K,NC_064042.1,334,334,0.501506,0.557872
6,WIDTH128_25K,NC_064034.1,279,279,0.504965,0.543827
7,WIDTH128_25K,NC_064041.1,259,259,0.505628,0.551438
8,WIDTH128_25K,NC_064042.1,334,334,0.502851,0.546282
9,WIDTH256_10K,NC_064034.1,279,279,0.502704,0.587056



Mean probe AUC:


,representation,baseline_score_ROC_AUC,linear_probe_ROC_AUC
0,WIDTH128_10K,0.503058,0.580426
1,WIDTH128_25K,0.504481,0.547182
2,WIDTH256_10K,0.504386,0.561944
3,WIDTH256_25K,0.504075,0.512400
4,WIDTH64_10K,0.504136,0.514316


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width256_25k_lowtp_scorematched_probe.csv
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width_training_scaling_lowtp_probe.csv


# Прямые сравнения

### Training-budget comparison
```text
WIDTH256 @10k  EXP040
vs
WIDTH256 @25k  EXP042
```

### Equal-budget width comparison
```text
WIDTH128 @25k  EXP041
vs
WIDTH256 @25k  EXP042
```

Это позволяет отделить:
- эффект количества шагов;
- эффект ширины.


In [17]:
# ============================================================
# REFERENCES — 256@10K AND 128@25K
# ============================================================

EXP040_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP040_WIDTH256_RF1029"
    / "run_001"
)

EXP041_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP041_WIDTH128_25K_RF1029"
    / "run_001"
)

EXP040_METRICS = (
    EXP040_RUN
    / "strong_validation_metrics.csv"
)

EXP041_METRICS = (
    EXP041_RUN
    / "strong_validation_metrics.csv"
)

print("WIDTH256 @25k — EXP042:")
display(metrics)
print()

comparisons = []

if EXP040_METRICS.is_file():
    exp040 = pd.read_csv(EXP040_METRICS)

    compare_10k_25k = (
        exp040[
            [
                "segment",
                "presence_AP_5dp",
                "presence_EPIC_Spearman_raw",
                "intensity_EPIC_Spearman_raw",
            ]
        ]
        .merge(
            metrics[
                [
                    "segment",
                    "presence_AP_5dp",
                    "presence_EPIC_Spearman_raw",
                    "intensity_EPIC_Spearman_raw",
                ]
            ],
            on="segment",
            suffixes=(
                "_256_10k",
                "_256_25k",
            ),
        )
    )

    compare_10k_25k["delta_AP"] = (
        compare_10k_25k["presence_AP_5dp_256_25k"]
        - compare_10k_25k["presence_AP_5dp_256_10k"]
    )

    print("WIDTH256 @10k vs @25k:")
    display(compare_10k_25k)

    path = (
        RUN_DIR
        / "width256_10k_vs_25k_metrics.csv"
    )
    compare_10k_25k.to_csv(
        path,
        index=False,
    )
    comparisons.append(str(path))

else:
    print("EXP040 metrics not found:", EXP040_METRICS)

if EXP041_METRICS.is_file():
    exp041 = pd.read_csv(EXP041_METRICS)

    compare_128_256_25k = (
        exp041[
            [
                "segment",
                "presence_AP_5dp",
                "presence_EPIC_Spearman_raw",
                "intensity_EPIC_Spearman_raw",
            ]
        ]
        .merge(
            metrics[
                [
                    "segment",
                    "presence_AP_5dp",
                    "presence_EPIC_Spearman_raw",
                    "intensity_EPIC_Spearman_raw",
                ]
            ],
            on="segment",
            suffixes=(
                "_128_25k",
                "_256_25k",
            ),
        )
    )

    compare_128_256_25k["delta_AP_256_minus_128"] = (
        compare_128_256_25k["presence_AP_5dp_256_25k"]
        - compare_128_256_25k["presence_AP_5dp_128_25k"]
    )

    print()
    print("WIDTH128 @25k vs WIDTH256 @25k:")
    display(compare_128_256_25k)

    path = (
        RUN_DIR
        / "width128_vs_width256_25k_metrics.csv"
    )
    compare_128_256_25k.to_csv(
        path,
        index=False,
    )
    comparisons.append(str(path))

else:
    print("EXP041 metrics not found:", EXP041_METRICS)

COMPARISON_FILES = comparisons


WIDTH256 @25k — EXP042:


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.075334,0.200920,0.170139,0.222120
1,NC_064041.1,13984,0.068389,0.235345,0.198931,0.249091
2,NC_064042.1,18692,0.082411,0.225596,0.185773,0.249896
3,overall,56205,0.075671,0.187675,0.141133,0.208023



WIDTH256 @10k vs @25k:


,segment,presence_AP_5dp_256_10k,presence_EPIC_Spearman_raw_256_10k,intensity_EPIC_Spearman_raw_256_10k,presence_AP_5dp_256_25k,presence_EPIC_Spearman_raw_256_25k,intensity_EPIC_Spearman_raw_256_25k,delta_AP
0,NC_064034.1,0.032077,0.185833,0.159889,0.075334,0.200920,0.222120,0.043257
1,NC_064041.1,0.028064,0.209003,0.185050,0.068389,0.235345,0.249091,0.040325
2,NC_064042.1,0.037105,0.196816,0.174672,0.082411,0.225596,0.249896,0.045307
3,overall,0.032517,0.170113,0.151181,0.075671,0.187675,0.208023,0.043154



WIDTH128 @25k vs WIDTH256 @25k:


,segment,presence_AP_5dp_128_25k,presence_EPIC_Spearman_raw_128_25k,intensity_EPIC_Spearman_raw_128_25k,presence_AP_5dp_256_25k,presence_EPIC_Spearman_raw_256_25k,intensity_EPIC_Spearman_raw_256_25k,delta_AP_256_minus_128
0,NC_064034.1,0.072823,0.201311,0.220176,0.075334,0.200920,0.222120,0.002510
1,NC_064041.1,0.070201,0.238797,0.244559,0.068389,0.235345,0.249091,-0.001812
2,NC_064042.1,0.080031,0.222173,0.247610,0.082411,0.225596,0.249896,0.002380
3,overall,0.074413,0.188400,0.205569,0.075671,0.187675,0.208023,0.001259


In [18]:
# ============================================================
# SAVE COMPACT RESULT JSON
# ============================================================

RESULT_JSON = (
    RUN_DIR
    / "EXP042_WIDTH256_25K_RESULT.json"
)

overall_metrics = (
    metrics
    .loc[
        metrics["segment"] == "overall"
    ]
    .iloc[0]
    .to_dict()
)

result_payload = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "hypothesis": (
            "WIDTH256 at 10k was undertrained; continuing the exact "
            "EXP040 optimizer/sampler trajectory to 25k improves "
            "natural-background ranking."
        ),
        "channels": 256,
        "theoretical_rf": 1029,
        "resumed_from": "CNN-EXP-040",
        "start_step": START_STEP,
        "steps": STEPS,
        "additional_steps": (
            STEPS - START_STEP
        ),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "only_intended_change": (
            "training budget 10k→25k; exact checkpoint resume"
        ),
    },

    "overall": {
        key: (
            int(value)
            if key == "positive_n"
            else float(value)
        )
        for key, value in overall_metrics.items()
        if key != "segment"
    },

    "references": {
        "width256_10k": {
            "experiment": "CNN-EXP-040",
            "channels": 256,
            "steps": 10_000,
        },
        "width128_25k": {
            "experiment": "CNN-EXP-041",
            "channels": 128,
            "steps": 25_000,
        },
    },

    "files": {
        "checkpoint": str(FINAL_CHECKPOINT),
        "history": str(HISTORY_PATH),
        "strong_validation_metrics": str(METRICS_PATH),
        "comparison_files": COMPARISON_FILES,
        "error_population_summary": str(COHORT_PATH),
        "positive_strength_summary": str(STRENGTH_PATH),
        "feature_probe_samples": (
            str(FEATURES_NPZ)
            if RUN_FEATURE_PROBE
            else None
        ),
        "feature_probe": (
            str(PROBE_PATH)
            if RUN_FEATURE_PROBE
            else None
        ),
        "lowtp_scorematched_probe": (
            str(LOWTP_PROBE_PATH)
            if RUN_FEATURE_PROBE
            else None
        ),
        "representation_scaling_comparison": (
            str(WIDTH_COMPARISON_PATH)
            if RUN_FEATURE_PROBE
            else None
        ),
    },

    "decision_questions": [
        "Did WIDTH256 AP improve materially from 10k to 25k?",
        "Is the change consistent across validation contigs?",
        "Does WIDTH256 @25k beat WIDTH128 @25k at equal training budget?",
        "Did presence/intensity Spearman improve?",
        "Did LOW_TP score-matched representation AUC improve with more training?",
        "Do the 25k curves justify going to 50k before LR decay?",
    ],
}

RESULT_JSON.write_text(
    json.dumps(
        result_payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("=" * 72)
print("EXP042 COMPLETE")
print("=" * 72)
print()
print("Send me:")
print(RESULT_JSON)

if RUN_FEATURE_PROBE:
    print(PROBE_PATH)
    print(LOWTP_PROBE_PATH)
    print(WIDTH_COMPARISON_PATH)


EXP042 COMPLETE

Send me:
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\EXP042_WIDTH256_25K_RESULT.json
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width256_25k_cross_contig_feature_probe.csv
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width256_25k_lowtp_scorematched_probe.csv
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\width_training_scaling_lowtp_probe.csv


In [20]:
# ============================================================
# OPTIONAL: REGISTER EXPERIMENT AFTER REVIEW
# ============================================================

SAVE_EXPERIMENT = True

FINAL_DECISION = "review"
FINAL_NOTES = (
    "WIDTH256 exact resume from EXP040 10k to 25k. "
    "LR, architecture, sampling and batch unchanged."
)

REPORT_METRICS_PATH = (
    RUN_DIR
    / "validation_metrics_report.csv"
)

report_metrics = (
    metrics[
        [
            "segment",
            "presence_AP_5dp",
            "presence_EPIC_Spearman_raw",
        ]
    ]
    .rename(
        columns={
            "presence_AP_5dp": "average_precision",
            "presence_EPIC_Spearman_raw": "epic_spearman",
        }
    )
)

report_metrics.to_csv(
    REPORT_METRICS_PATH,
    index=False,
)

EXP040_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP040_WIDTH256_RF1029"
    / "run_001"
)

REFERENCE_REPORT_PATH = (
    EXP040_RUN
    / "validation_metrics_report.csv"
)

if not REFERENCE_REPORT_PATH.is_file():
    exp040_strong = (
        EXP040_RUN
        / "strong_validation_metrics.csv"
    )

    if exp040_strong.is_file():
        exp040_metrics = pd.read_csv(exp040_strong)

        (
            exp040_metrics[
                [
                    "segment",
                    "presence_AP_5dp",
                    "presence_EPIC_Spearman_raw",
                ]
            ]
            .rename(
                columns={
                    "presence_AP_5dp": "average_precision",
                    "presence_EPIC_Spearman_raw": "epic_spearman",
                }
            )
            .to_csv(
                REFERENCE_REPORT_PATH,
                index=False,
            )
        )

CARD_PATH = (
    PROJECT_ROOT
    / "cnn_exp"
    / f"{EXPERIMENT_ID}.md"
)

CARD_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not CARD_PATH.exists():
    card_text = (
        "---\n"
        f"id: {EXPERIMENT_ID}\n"
        "type: experiment\n"
        "experiment_type: cnn-candidate\n"
        "status: completed\n"
        "decision: review\n"
        "date: 2026-09-25\n"
        "seed: 42\n"
        "split_version: contig_holdout_v1\n"
        "primary_metric: average_precision\n"
        "secondary_metric: epic_spearman\n"
        "reference_experiment: CNN-EXP-040\n"
        "tags:\n"
        "  - ml/experiment\n"
        "  - ml/cnn\n"
        "  - ml/training-budget\n"
        "---\n\n"
        f"# {EXPERIMENT_ID} — WIDTH256 @25k RF1029\n\n"
        "← [[cnn_exp/_index.md|Реестр CNN]]\n\n"
        "## Pre-registration\n\n"
        "- **Гипотеза:** WIDTH256 @10k недообучен; continuation до 25k "
        "улучшит natural-background ranking.\n"
        "- **Единственное изменение:** training budget 10k → 25k.\n"
        "- **Resume:** exact checkpoint EXP040 @10k с optimizer и sampler RNG.\n"
        "- **Что заморожено:** WIDTH256, RF1029, LR=1e-3, batch=4, "
        "split, generator logic, targets, loss.\n"
        "- **Reference:** [[cnn_exp/CNN-EXP-040.md|CNN-EXP-040]].\n\n"
        "## Автоматический отчёт\n\n"
        "<!-- auto:cnn-experiment-report:start -->\n\n"
        "<!-- auto:cnn-experiment-report:end -->\n\n"
        "## Ручные заметки\n\n"
        "-\n"
    )

    CARD_PATH.write_text(
        card_text,
        encoding="utf-8",
    )

print("Reporter metrics:", REPORT_METRICS_PATH)
print("Reference reporter metrics:", REFERENCE_REPORT_PATH)
print("Card:", CARD_PATH)

if SAVE_EXPERIMENT:

    from ml_project.cnn_experiment_report import (
        finalize_cnn_experiment,
    )

    assert REFERENCE_REPORT_PATH.is_file(), (
        f"Reference reporter metrics missing: {REFERENCE_REPORT_PATH}"
    )

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id=EXPERIMENT_ID,
        title="WIDTH256 @25k RF1029",
        hypothesis=(
            "Continuing the exact WIDTH256 EXP040 trajectory "
            "from 10k to 25k improves natural-background ranking"
        ),
        notebook_path=(
            "notebooks/experiments/CNN_experiments/"
            "CNN_EXP042_WIDTH256_25K_RF1029_STRONG_VALIDATION.ipynb"
        ),
        artifact_dir=RUN_DIR,
        training_history=HISTORY_PATH,
        candidate_metrics=REPORT_METRICS_PATH,
        reference_experiment="CNN-EXP-040",
        reference_metrics=REFERENCE_REPORT_PATH,
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )

    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)

else:
    print(
        "SAVE_EXPERIMENT=False. "
        "Сначала пришли EXP042_WIDTH256_25K_RESULT.json "
        "и probe CSV."
    )


Reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\validation_metrics_report.csv
Reference reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP040_WIDTH256_RF1029\run_001\validation_metrics_report.csv
Card: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-042.md
Obsidian: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-042.md
Summary: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\experiment_summary.json
Plots: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP042_WIDTH256_25K_RF1029\run_001\plots
